In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import pandas as pd
from sklearn.base import clone
import time
import itertools
from sklearn.metrics import confusion_matrix, recall_score
from sklearn.utils import resample
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
def evaluate_basic_classifier(y_true, y_pred, y_score=None):
    from sklearn.metrics import (
        confusion_matrix, accuracy_score, precision_score, recall_score,
        f1_score, matthews_corrcoef, jaccard_score, roc_auc_score
    )
    import numpy as np
    import pandas as pd

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    accuracy = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    precision = precision_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    jaccard = jaccard_score(y_true, y_pred)
    g_measure = np.sqrt(recall * specificity)


    auc = roc_auc_score(y_true, y_score) if y_score is not None else np.nan

    metrics = {
        "Metric": [
            "Accuracy",  "MCC", "F1 Score","G-Measure", "AUC",
            "Precision", "Recall (Sensitivity)", "Specificity", "Jaccard Index"
        ],
        "Value": [
            accuracy, mcc, f1, g_measure, auc,
            precision, recall, specificity, jaccard
        ]
    }
    
    print("\n===== Classification Performance Metrics =====")
    for metric, value in zip(metrics["Metric"], metrics["Value"]):
        print(f"{metric}: {value:.4f}")
    print("=============================================\n")
    return pd.DataFrame(metrics)


In [ ]:

# Function to get out-of-fold predictions using selected first-layer classifiers
def get_oof_with_selected_models(x_train, y_train, x_test, selected_models, model_names, n_repeats=5):
    # Number of selected models
    n_selected = len(selected_models)
    
    # Final matrices to store results
    proba_matrix = np.zeros((x_train.shape[0], n_selected * 2))  # OOF training predictions
    proba_ind_mean = np.zeros((x_test.shape[0], n_selected * 2))  # Test set predictions (averaged)
    
    # Store all iterations' test predictions for final averaging
    all_test_predictions = np.zeros((n_repeats, x_test.shape[0], n_selected * 2))
    
    for iteration in range(n_repeats):
        # Create a new KFold split for each iteration
        kf = KFold(n_splits=5, shuffle=True, random_state=23 + iteration)
        
        # Temporary matrix for this iteration's fold predictions
        iter_proba_matrix = np.zeros((x_train.shape[0], n_selected * 2))
        iter_test_predictions = np.zeros((5, x_test.shape[0], n_selected * 2))
        
        print(f"Starting iteration {iteration+1}/{n_repeats}")
        
        for fold_idx, (train_index, test_index) in enumerate(kf.split(x_train)):
            kf_x_train = x_train.iloc[train_index]
            kf_y_train = y_train[train_index]
            kf_x_test = x_train.iloc[test_index]
            
            for idx, model in enumerate(selected_models):
                # Clone model to avoid data leakage between folds/iterations
                current_model = clone(model)
                
                # Fit the model
                current_model.fit(kf_x_train, kf_y_train)
                
                # Generate predictions
                if hasattr(current_model, "predict_proba"):
                    proba_e = current_model.predict_proba(kf_x_test)
                    proba_ind = current_model.predict_proba(x_test)
                else:  # For regression models like LassoCV
                    proba_e = current_model.predict(kf_x_test).reshape(-1, 1)
                    proba_ind = current_model.predict(x_test).reshape(-1, 1)
                    # Convert regression output to pseudo-probabilities
                    proba_e = np.hstack([1 - proba_e, proba_e])  
                    proba_ind = np.hstack([1 - proba_ind, proba_ind])
                
                # Store predictions for this fold
                iter_proba_matrix[test_index, idx * 2 : (idx + 1) * 2] = proba_e#this is result of 5-folds CV
                iter_test_predictions[fold_idx, :, idx * 2 : (idx + 1) * 2] = proba_ind#this is result of ind test data
            
            print(f"  Iteration {iteration+1}, fold {fold_idx+1} completed")
        
        # Accumulate train predictions (average them later)
        proba_matrix += iter_proba_matrix
        
        # Store this iteration's test predictions
        all_test_predictions[iteration] = iter_test_predictions.mean(axis=0)
        
        # Save this iteration's results
        combo_name = "-".join([name[:3] for name in model_names])
        #pd.DataFrame(iter_proba_matrix).to_csv(f"./proba_matrix_train_{combo_name}_iter_{iteration+1}.csv", index=False)
        #pd.DataFrame(iter_test_predictions.mean(axis=0)).to_csv(f"./proba_matrix_test_{combo_name}_iter_{iteration+1}.csv", index=False)
    
    # Average the train predictions across all iterations
    proba_matrix = proba_matrix / n_repeats
    
    # Average the test predictions across all iterations
    proba_ind_mean = all_test_predictions.mean(axis=0)
    
    df_proba_train = pd.DataFrame(proba_matrix)
    df_proba_test = pd.DataFrame(proba_ind_mean)
    df_proba_train.to_csv("proba_matrix.csv", index=False)
    df_proba_test.to_csv("proba_ind_mean.csv", index=False)
  
    return proba_matrix, proba_ind_mean


In [ ]:

#feature importance
xtest = pd.read_csv('test_data_features.csv',index_col=0)
ytest = xtest.pop('labels')

## Prepare training data
xtrain = pd.read_csv('train_data_features.csv',index_col=0)
ytrain = xtrain.pop('labels')

ytrain = ytrain.to_numpy()
ytest  = ytest.to_numpy()


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
import numpy as np
import pandas as pd
# Main execution
def main():
    seed=42
       
    all_models = [
        SVC(kernel="rbf", C=1, probability=True),
     
        KNeighborsClassifier(n_neighbors=50)
    ]

    model_names = ["SVC", "KNN"]

    # Define second-layer models
    second_layer_models = [
        KNeighborsClassifier(n_neighbors=50),
    ]

    second_layer_names = ["KNN"]
    
    # Generate all combinations of 5 first-layer models
    combinations = list(itertools.combinations(range(len(all_models)), 2))
    
    best_Accuracy = 0
    best_first_layer = None
    best_second_layer = None
    best_metrics = None
    
    results = []

    # Test each combination with each second-layer model
    for combo_idx, combo in enumerate(combinations):
        selected_models = [all_models[i] for i in combo]
        selected_model_names = [model_names[i] for i in combo]
        
        combo_name = "-".join([name[:3] for name in selected_model_names])
        print(f"\n\n===== Testing Combination {combo_idx+1}/{len(combinations)}: {combo_name} =====")
        
        # Get out-of-fold predictions for this combination
        start_time = time.time()
        proba_matrix, proba_ind_mean = get_oof_with_selected_models(
            xtrain, ytrain, xtest, selected_models, selected_model_names, n_repeats=5
        )
        end_time = time.time()
        all_time=(end_time - start_time)/60
        print(f"Model training time: {all_time:.2f} minutes")
        
        # Prepare data for second-layer models
        # Add original features to meta-features
        xtrain_num_columns = proba_matrix.shape[1]
        xtrain_array_columns = [f"Meta_{i+1}" for i in range(xtrain_num_columns)]
        df_array = pd.DataFrame(proba_matrix, columns=xtrain_array_columns)
        df_array.index = xtrain.index
        xtrain_enhanced = pd.concat([xtrain, df_array], axis=1) #concatenate together
       
        
        # Same for test data
        xtest_num_columns = proba_ind_mean.shape[1]
        xtest_array_columns = [f"Meta_{i+1}" for i in range(xtest_num_columns)]
        xtest_df_array = pd.DataFrame(proba_ind_mean, columns=xtest_array_columns)
        xtest_df_array.index = xtest.index
        xtest_enhanced = pd.concat([xtest, xtest_df_array], axis=1) #concatenate together

       
        # Set up 5-fold cross-validation
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        for model_idx, model in enumerate(second_layer_models):
            second_name = second_layer_names[model_idx]
            print(f"\nTesting with {second_name} as second-layer classifier")
        
            try:
                oof_preds = np.zeros_like(ytrain, dtype=float)
                test_preds = []
        
                for fold_idx, (train_idx, val_idx) in enumerate(cv.split(xtrain_enhanced, ytrain)):
                    print(f"Fold {fold_idx+1}/5")
        
                    x_tr, y_tr = xtrain_enhanced.iloc[train_idx], ytrain[train_idx]
                    x_val, y_val = xtrain_enhanced.iloc[val_idx], ytrain[val_idx]
        
                    model_fold = clone(model)
                    model_fold.fit(x_tr, y_tr)
        
                    # Predict
                    if hasattr(model_fold, "predict_proba"):
                        val_prob = model_fold.predict_proba(x_val)[:, 1]
                        test_prob = model_fold.predict_proba(xtest_enhanced)[:, 1]
                    else:
                        val_prob = model_fold.predict(x_val)
                        test_prob = model_fold.predict(xtest_enhanced)
        
                    oof_preds[val_idx] = val_prob
                    test_preds.append(test_prob)
        
                # Final predictions
               
                test_preds = np.array(test_preds)
                
                oof_final  = (oof_preds >= 0.5).astype(int)
                test_final = (test_preds.mean(axis=0) >= 0.5).astype(int)

        
                # Evaluation
                print(f"for off_final:\n{oof_final}")
                print(f"for test_final:\n{test_final}")
                print("for training set")
                train_metrics_df = evaluate_basic_classifier(ytrain,oof_final,y_score=oof_preds )
                print("for test set")
                
                test_score = test_preds.mean(axis=0)
                
            
                test_output_df = pd.DataFrame({
                    "y_true": ytest,
                    "y_pred": test_final,
                    "y_score": test_score
                })
                
                test_output_df.index = xtest.index
                
                test_output_df.to_csv("test_predictions.csv", index=False)
               
                test_metrics_df = evaluate_basic_classifier(
                    ytest,
                    test_final,
                    y_score=test_score
                )
        
                train_Accuracy = train_metrics_df.loc[train_metrics_df['Metric'] == "Accuracy", "Value"].values[0]
                test_Accuracy = test_metrics_df.loc[test_metrics_df['Metric'] == "Accuracy", "Value"].values[0]
        
                result_row = {
                    'First_Layer_Models': combo_name,
                    'Second_Layer_Model': second_name,
                    'Train_Accuracy': train_Accuracy,
                    'Test_Accuracy': test_Accuracy
                }
        
                
                for i, metric in enumerate(train_metrics_df['Metric']):
                    result_row[f"Train_{metric}"] = train_metrics_df['Value'].iloc[i]
                for i, metric in enumerate(test_metrics_df['Metric']):
                    result_row[f"Test_{metric}"] = test_metrics_df['Value'].iloc[i]
        
                results.append(result_row)
        
                if test_Accuracy > best_Accuracy:
                    best_Accuracy = test_Accuracy
                    best_first_layer = combo_name
                    best_second_layer = second_name
                    best_metrics = test_metrics_df.copy()
                    print(f"New best: {best_first_layer} + {best_second_layer} | Accuracy = {best_Accuracy:.4f}")
        
            except Exception as e:
                print(f"Error with {second_name}: {str(e)}")
                continue

        
    # Save
        results_df = pd.DataFrame(results)
        results_df.to_csv("AMPScanner_svmlrknn_results.csv", index=False)
        
        print("\n\n===== BEST COMBINATION =====")
        print(f"First Layer Models: {best_first_layer}")
        print(f"Second Layer Model: {best_second_layer}")
        

In [ ]:

import warnings
import pandas as pd
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

start_time = time.time()
if __name__ == "__main__":
    main()

end_time = time.time()

all_time=(end_time - start_time)/60
print(f"Whole time: {all_time:.2f} minutes")
